# LASSO Regression for Five Drug Synergy Datasets

This notebook trains **one independent LASSO regression model per dataset**. It uses the 330 `DRS_feature_*` columns plus `drugname1`, `drugname2`, and `cellline` (when present) to predict `synergy_score`.

**Before running:** Put this notebook in the same folder as the five CSV files, or change `DATA_DIR` in the next cell.

The uploaded Almanac and OncologyScreen CSVs have an extra quote around the entire header/data row. The loader below handles that formatting issue. It also removes duplicate rows, cleans missing values, scales numeric features, one-hot encodes categorical columns, and reports MAE, RMSE, and R².

LASSO is a linear model with L1 regularization. Its R² is an evaluation result, not a guaranteed target; do not change the test set to force a higher score.

In [1]:
# Cell 1: Install packages if needed
# Run this cell once if pandas / scikit-learn / joblib are not installed.
%pip install -q pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2: Imports and dataset locations
import csv
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# Keep the five CSV files in the same folder as this notebook.
DATA_DIR = Path(".")
OUTPUT_DIR = Path("lasso_results")
OUTPUT_DIR.mkdir(exist_ok=True)

DATASETS = {
    "Almanac": DATA_DIR / "almanac_final_without_last_score.csv",
    "DrugComb": DATA_DIR / "DrugComb_LINCS_PRISM_final_without_last_score.csv",
    "DrugCombDB": DATA_DIR / "DrugCombDBLINCS_PRISM_final_without_last_score.csv",
    "OncologyScreen": DATA_DIR / "OncologyScreenLINCS_PRISM_final_without_last_score.csv",
    "Oneil": DATA_DIR / "OneilLINCS_PRISM_final_without_last_score.csv",
}

TARGET_CANDIDATES = ["synergy_score", "score", "SYNERGY_DELTA_AUC", "Synergy_DELTA_AUC"]
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Output folder:", OUTPUT_DIR.resolve())
for name, path in DATASETS.items():
    print(name, "->", path, "| exists:", path.exists())

Output folder: /Users/sathvikareddy/lasso/lasso_results
Almanac -> almanac_final_without_last_score.csv | exists: True
DrugComb -> DrugComb_LINCS_PRISM_final_without_last_score.csv | exists: True
DrugCombDB -> DrugCombDBLINCS_PRISM_final_without_last_score.csv | exists: True
OncologyScreen -> OncologyScreenLINCS_PRISM_final_without_last_score.csv | exists: True
Oneil -> OneilLINCS_PRISM_final_without_last_score.csv | exists: True


In [11]:
# Cell 3: Robust CSV loading and cleaning function

def load_and_clean_dataset(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(
            f"Cannot find {file_path}. Put the CSV beside this notebook or update DATA_DIR."
        )

    # First try normal CSV parsing. If the file is wrapped in quotes as a whole,
    # fall back to parsing with quoting disabled.
    df = pd.read_csv(file_path, low_memory=False)
    if df.shape[1] == 1:
        df = pd.read_csv(file_path, quoting=csv.QUOTE_NONE, engine="python")

    # Remove accidental quote characters and surrounding spaces in column names.
    df.columns = [str(col).strip().strip('"').strip("'") for col in df.columns]

    # Remove surrounding quotes/spaces from text values.
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip().str.strip('"').str.strip("'")
        df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan})

    # Identify the target column, allowing common naming variations.
    target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
    if target_col is None:
        raise ValueError(
            f"No synergy target column found in {file_path.name}. "
            f"Available final columns: {list(df.columns[-10:])}"
        )

    # Remove exact duplicate rows.
    before = len(df)
    df = df.drop_duplicates().copy()
    duplicates_removed = before - len(df)

    # Convert target to numeric and remove rows without a valid target.
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col]).copy()

    # Remove rows where the target is infinite.
    df = df[np.isfinite(df[target_col])].copy()

    # Find DRS feature columns (expected: DRS_feature_1 ... DRS_feature_330).
    drs_cols = [c for c in df.columns if re.fullmatch(r"DRS_feature_\\d+", str(c))]
    for col in drs_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Include available drug/cell-line identifiers as categorical predictors.
    categorical_cols = [c for c in ["drugname1", "drugname2", "cellline"] if c in df.columns]

    # Keep only rows with at least one usable predictor.
    feature_cols = drs_cols + categorical_cols
    if not feature_cols:
        raise ValueError(f"No DRS_feature_* or drug/cell-line columns found in {file_path.name}")

    # Drop features that are entirely missing.
    drs_cols = [c for c in drs_cols if df[c].notna().any()]
    categorical_cols = [
        c for c in categorical_cols
        if df[c].notna().any()
    ]
    feature_cols = drs_cols + categorical_cols

    # Ensure categorical missing values are handled by the pipeline.
    # Return cleaned data and metadata for training.
    print(f"Loaded: {file_path.name}")
    print(f"Rows after cleaning: {len(df):,} | Duplicates removed: {duplicates_removed:,}")
    print(f"DRS features: {len(drs_cols)} | Categorical features: {categorical_cols}")
    print(f"Target column: {target_col}")
    return df, feature_cols, drs_cols, categorical_cols, target_col

In [12]:
# Cell 4: Train and evaluate one LASSO model

def train_lasso_for_dataset(dataset_name, file_path, alpha=0.01):
    df, feature_cols, numeric_cols, categorical_cols, target_col = load_and_clean_dataset(file_path)

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    # Use a reproducible hold-out test set.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # Numeric DRS features: median imputation + standardization.
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Drug and cell-line names: most-frequent imputation + one-hot encoding.
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("numeric", numeric_pipeline, numeric_cols))
    if categorical_cols:
        transformers.append(("categorical", categorical_pipeline, categorical_cols))

    preprocessor = ColumnTransformer(transformers=transformers)

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("lasso", Lasso(alpha=alpha, max_iter=30000, tol=0.001))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print("\\n" + "=" * 55)
    print(f"{dataset_name} - LASSO RESULTS")
    print("=" * 55)
    print(f"Training rows : {len(X_train):,}")
    print(f"Testing rows  : {len(X_test):,}")
    print(f"MAE           : {mae:.4f}")
    print(f"RMSE          : {rmse:.4f}")
    print(f"R2 Score      : {r2:.4f}")
    print(f"R2 Percentage: {r2 * 100:.2f}%")

    # Save the complete preprocessing + model pipeline.
    model_path = OUTPUT_DIR / f"{dataset_name.lower()}_lasso_model.joblib"
    joblib.dump(model, model_path)
    print("Saved model   :", model_path)

    # Save a small prediction sample for inspection.
    prediction_sample = pd.DataFrame({
        "actual_synergy": y_test.iloc[:20].to_numpy(),
        "predicted_synergy": predictions[:20]
    })
    prediction_path = OUTPUT_DIR / f"{dataset_name.lower()}_lasso_predictions.csv"
    prediction_sample.to_csv(prediction_path, index=False)

    return {
        "Dataset": dataset_name,
        "Rows": len(df),
        "Features": len(feature_cols),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "R2_Percentage": r2 * 100,
        "Model_File": str(model_path)
    }

## Almanac dataset

In [29]:
# Cell: Train LASSO on Almanac
result_almanac = train_lasso_for_dataset(
    "Almanac",
    DATASETS["Almanac"],
    alpha=0.01
)
result_almanac

Loaded: almanac_final_without_last_score.csv
Rows after cleaning: 29,251 | Duplicates removed: 0
DRS features: 0 | Categorical features: ['drugname1', 'drugname2', 'cellline']
Target column: synergy_score
\n=======================================================
Almanac - LASSO RESULTS
Training rows : 23,400
Testing rows  : 5,851
MAE           : 35.8072
RMSE          : 48.5482
R2 Score      : 0.2444
R2 Percentage: 24.44%
Saved model   : lasso_results/almanac_lasso_model.joblib


{'Dataset': 'Almanac',
 'Rows': 29251,
 'Features': 3,
 'MAE': 35.80723501578264,
 'RMSE': np.float64(48.54822804923487),
 'R2': 0.24437893224788831,
 'R2_Percentage': 24.43789322478883,
 'Model_File': 'lasso_results/almanac_lasso_model.joblib'}

## DrugComb dataset

In [6]:
# Cell: Train LASSO on DrugComb
result_drugcomb = train_lasso_for_dataset(
    "DrugComb",
    DATASETS["DrugComb"],
    alpha=0.01
)
result_drugcomb

Loaded: DrugComb_LINCS_PRISM_final_without_last_score.csv
Rows after cleaning: 17,139 | Duplicates removed: 0
DRS features: 0 | Categorical features: ['drugname1', 'drugname2', 'cellline']
Target column: synergy_score
\n=======================================================
DrugComb - LASSO RESULTS
Training rows : 13,711
Testing rows  : 3,428
MAE           : 3.4130
RMSE          : 4.7125
R2 Score      : 0.2303
R2 Percentage: 23.03%
Saved model   : lasso_results/drugcomb_lasso_model.joblib


{'Dataset': 'DrugComb',
 'Rows': 17139,
 'Features': 3,
 'MAE': 3.4130025608212233,
 'RMSE': np.float64(4.71254700535099),
 'R2': 0.23025310027735113,
 'R2_Percentage': 23.025310027735113,
 'Model_File': 'lasso_results/drugcomb_lasso_model.joblib'}

## DrugCombDB dataset

In [7]:
# Cell: Train LASSO on DrugCombDB
result_drugcombdb = train_lasso_for_dataset(
    "DrugCombDB",
    DATASETS["DrugCombDB"],
    alpha=0.01
)
result_drugcombdb

Loaded: DrugCombDBLINCS_PRISM_final_without_last_score.csv
Rows after cleaning: 17,139 | Duplicates removed: 0
DRS features: 0 | Categorical features: ['drugname1', 'drugname2', 'cellline']
Target column: synergy_score
\n=======================================================
DrugCombDB - LASSO RESULTS
Training rows : 13,711
Testing rows  : 3,428
MAE           : 3.4130
RMSE          : 4.7125
R2 Score      : 0.2303
R2 Percentage: 23.03%
Saved model   : lasso_results/drugcombdb_lasso_model.joblib


{'Dataset': 'DrugCombDB',
 'Rows': 17139,
 'Features': 3,
 'MAE': 3.4130025608212233,
 'RMSE': np.float64(4.71254700535099),
 'R2': 0.23025310027735113,
 'R2_Percentage': 23.025310027735113,
 'Model_File': 'lasso_results/drugcombdb_lasso_model.joblib'}

## OncologyScreen dataset

In [26]:
# Cell: Train LASSO on OncologyScreen
result_oncologyscreen = train_lasso_for_dataset(
    "OncologyScreen",
    DATASETS["OncologyScreen"],
    alpha=0.01
)
result_oncologyscreen

Loaded: OncologyScreenLINCS_PRISM_final_without_last_score.csv
Rows after cleaning: 1,160 | Duplicates removed: 0
DRS features: 0 | Categorical features: ['drugname1', 'drugname2', 'cellline']
Target column: synergy_score
\n=======================================================
OncologyScreen - LASSO RESULTS
Training rows : 928
Testing rows  : 232
MAE           : 14.8601
RMSE          : 20.1359
R2 Score      : 0.3008
R2 Percentage: 30.08%
Saved model   : lasso_results/oncologyscreen_lasso_model.joblib


{'Dataset': 'OncologyScreen',
 'Rows': 1160,
 'Features': 3,
 'MAE': 14.86011266098056,
 'RMSE': np.float64(20.13591933419956),
 'R2': 0.3008149872436543,
 'R2_Percentage': 30.08149872436543,
 'Model_File': 'lasso_results/oncologyscreen_lasso_model.joblib'}

## Oneil dataset

In [31]:
# Cell: Train LASSO on Oneil
result_oneil = train_lasso_for_dataset(
    "Oneil",
    DATASETS["Oneil"],
    alpha=0.01
)
result_oneil

Loaded: OneilLINCS_PRISM_final_without_last_score.csv
Rows after cleaning: 2,714 | Duplicates removed: 0
DRS features: 0 | Categorical features: ['drugname1', 'drugname2', 'cellline']
Target column: synergy_score
\n=======================================================
Oneil - LASSO RESULTS
Training rows : 2,171
Testing rows  : 543
MAE           : 14.2243
RMSE          : 18.8673
R2 Score      : 0.2892
R2 Percentage: 28.92%
Saved model   : lasso_results/oneil_lasso_model.joblib


{'Dataset': 'Oneil',
 'Rows': 2714,
 'Features': 3,
 'MAE': 14.22425462531943,
 'RMSE': np.float64(18.867281711128104),
 'R2': 0.28922462227543555,
 'R2_Percentage': 28.922462227543555,
 'Model_File': 'lasso_results/oneil_lasso_model.joblib'}

In [17]:
# Final cell: Compare all five dataset results
results = [
    result_almanac,
    result_drugcomb,
    result_drugcombdb,
    result_oncologyscreen,
    result_oneil
]

results_df = pd.DataFrame(results)
display(results_df.round(4))

results_df.to_csv(OUTPUT_DIR / "lasso_all_dataset_metrics.csv", index=False)
print("Metrics saved to:", OUTPUT_DIR / "lasso_all_dataset_metrics.csv")

,Dataset,Rows,Features,MAE,RMSE,R2,R2_Percentage,Model_File
0,Almanac,29251,3,35.8072,48.5482,0.2444,24.4379,lasso_results/almanac_lasso_model.joblib
1,DrugComb,17139,3,3.4130,4.7125,0.2303,23.0253,lasso_results/drugcomb_lasso_model.joblib
2,DrugCombDB,17139,3,3.4130,4.7125,0.2303,23.0253,lasso_results/drugcombdb_lasso_model.joblib
3,OncologyScreen,1160,3,14.8578,20.4424,0.2794,27.9367,lasso_results/oncologyscreen_lasso_model.joblib
4,Oneil,2714,3,14.2582,18.8867,0.2878,28.7759,lasso_results/oneil_lasso_model.joblib


Metrics saved to: lasso_results/lasso_all_dataset_metrics.csv


## Notes

- Each dataset gets its **own train/test split and its own LASSO model**. The datasets are not merged.
- The model uses DRS gene-expression signature features and, where available, drug names and cell-line identifiers.
- `alpha=0.01` controls LASSO regularization. If the model underfits, test a smaller alpha such as `0.001`; if it overfits, test a larger alpha such as `0.1` or `1.0`. Compare using the same split.
- A negative R² means the model performs worse than predicting the test-set mean. This can happen and should be reported honestly.
- This notebook evaluates prediction on a random held-out split. For a stronger generalization test, use a group-based split that keeps related drug pairs or cell lines out of the training set.